# R Master v1 · Drive 持久缓存云跑

这个版本解决两个问题：

- **不再每次上传 Mona**：第一次运行时会把源 `.blend` 存到你自己的 Google Drive。以后新 Colab 会话直接读取。
- **Blender 也缓存**：官方 Blender 4.4.3 压缩包会保存到 Drive，后续只需本地解压，不再重复走公网下载。

## 手机上怎么用

1. 点 **运行全部 / Run all**。
2. Google Drive 要求授权时，点允许。
3. **只有第一次**会要求选择一个源文件：
   - 最好选原始 `Mona.blend`；
   - 如果原始文件不方便，也可以选上次的 `R_Master_v0_Result.zip`，它会自动取里面的 `R_Master_Align_v0_PREVIEW.blend`，原始 Mona 仍在那个工程里。
4. 以后再跑 v1/v2，只要 Drive 里缓存还在，就不会再要求上传。

## v1 会做什么

- 在 Mona 副本上临时关掉会干扰比例预览的 IK / Copy Transforms / Limit Scale 等约束；
- 用继承缩放补偿重新计算腿、脚、手臂、躯干和颈部；
- 把髋关节宽度纳入 **预览**；
- 只渲染 `Mona_Main` 的灰模正面 / 侧面 / 3/4，专门看结构，不让衣服和控制器挡住；
- 生成新的预览 `.blend` 和验证报告；
- 重文件留在 Drive，只自动下载一个很小的 review ZIP。

> 仍然不会修改原始 Mona、不会碰 RED 正式页面、不会烘焙最终 Rest Pose、不会导出最终 VRM。


In [ ]:
# R Master v1 · Persistent Google Drive runner
from google.colab import drive, files
from IPython.display import display, Image, Markdown
from pathlib import Path
import os, shutil, subprocess, urllib.request, zipfile, json, time, hashlib

BLENDER_VERSION = "4.4.3"
BLENDER_URL = "https://download.blender.org/release/Blender4.4/blender-4.4.3-linux-x64.tar.xz"
SCRIPT_COMMIT = "11e53b0bfb291e0f3140da74deb0bc28e2420cf4"
SCRIPT_URL = f"https://raw.githubusercontent.com/hexiangyu481-commits/-erdan-lab-mobile/{SCRIPT_COMMIT}/red-r-master/blender/R_Master_BuildPreview_v1.py"

LOCAL = Path('/content/r_master_v1')
LOCAL_OUT = LOCAL / 'output'
LOCAL_ARCHIVE = LOCAL / f'blender-{BLENDER_VERSION}-linux-x64.tar.xz'
LOCAL_BLENDER_DIR = LOCAL / f'blender-{BLENDER_VERSION}-linux-x64'
LOCAL_SCRIPT = LOCAL / 'R_Master_BuildPreview_v1.py'
LOCAL_SOURCE = LOCAL / 'Mona_Source.blend'

LOCAL.mkdir(parents=True, exist_ok=True)

print('R Master v1 · Drive 持久缓存云跑')
print('① 挂载你的 Google Drive…')
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/R_Master')
DRIVE_CACHE = DRIVE_ROOT / 'cache'
DRIVE_V1 = DRIVE_ROOT / 'v1' / 'latest'
DRIVE_CACHE.mkdir(parents=True, exist_ok=True)
DRIVE_V1.mkdir(parents=True, exist_ok=True)

DRIVE_SOURCE = DRIVE_CACHE / 'Mona_Source.blend'
DRIVE_ARCHIVE = DRIVE_CACHE / LOCAL_ARCHIVE.name


def sha256(path, chunk=1024*1024):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        while True:
            b=f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()


def ensure_source():
    if DRIVE_SOURCE.exists() and DRIVE_SOURCE.stat().st_size > 50*1024*1024:
        print(f'✓ 已找到 Drive 缓存源文件：{DRIVE_SOURCE.name} · {DRIVE_SOURCE.stat().st_size/1024/1024:.1f} MiB')
        return

    print('\nDrive 里还没有 Mona 源文件。只需要这一次上传。')
    print('可选：Mona.blend 或上次下载的 R_Master_v0_Result.zip')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('没有收到文件。重新运行即可。')

    names = list(uploaded.keys())
    blends = [n for n in names if n.lower().endswith('.blend')]
    zips = [n for n in names if n.lower().endswith('.zip')]

    temp = LOCAL / 'incoming'
    if temp.exists(): shutil.rmtree(temp)
    temp.mkdir(parents=True, exist_ok=True)

    if len(blends) == 1:
        p = temp / blends[0]
        p.write_bytes(uploaded[blends[0]])
        shutil.copy2(p, DRIVE_SOURCE)
    elif len(zips) == 1:
        zpath = temp / zips[0]
        zpath.write_bytes(uploaded[zips[0]])
        with zipfile.ZipFile(zpath, 'r') as z:
            candidates = [n for n in z.namelist() if n.lower().endswith('.blend')]
            if not candidates:
                raise RuntimeError('这个 ZIP 里没有 .blend 文件。')
            # Prefer the known v0 preview file, otherwise largest .blend member.
            preferred = [n for n in candidates if n.endswith('R_Master_Align_v0_PREVIEW.blend')]
            choice = preferred[0] if preferred else max(candidates, key=lambda n: z.getinfo(n).file_size)
            with z.open(choice) as src, open(DRIVE_SOURCE, 'wb') as dst:
                shutil.copyfileobj(src, dst, 1024*1024)
    else:
        raise RuntimeError(f'请只选 1 个 .blend 或 1 个 .zip。当前收到：{names}')

    if DRIVE_SOURCE.stat().st_size < 50*1024*1024:
        raise RuntimeError('缓存源文件尺寸异常，未继续。')
    print(f'✓ Mona 已永久缓存到 Drive：{DRIVE_SOURCE} · {DRIVE_SOURCE.stat().st_size/1024/1024:.1f} MiB')


ensure_source()

print('\n② 把 Mona 从 Drive 快速复制到 Colab 本地 SSD…')
shutil.copy2(DRIVE_SOURCE, LOCAL_SOURCE)
print(f'✓ 本地源：{LOCAL_SOURCE.stat().st_size/1024/1024:.1f} MiB')

print('\n③ 准备 Blender 4.4.3…')
subprocess.run(['apt-get','update','-qq'], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
subprocess.run(
    ['apt-get','install','-y','-qq','xvfb','libgl1','libx11-6','libxi6','libxrender1','libxfixes3','libxkbcommon0','libsm6'],
    check=True, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT
)

if not DRIVE_ARCHIVE.exists() or DRIVE_ARCHIVE.stat().st_size < 100*1024*1024:
    print('首次缓存 Blender 官方压缩包…')
    subprocess.run(['wget','-q','--show-progress','-O',str(LOCAL_ARCHIVE),BLENDER_URL], check=True)
    shutil.copy2(LOCAL_ARCHIVE, DRIVE_ARCHIVE)
    print(f'✓ Blender 压缩包已缓存到 Drive：{DRIVE_ARCHIVE.name}')
else:
    print('✓ Drive 里已有 Blender 缓存，跳过公网下载')
    shutil.copy2(DRIVE_ARCHIVE, LOCAL_ARCHIVE)

if LOCAL_BLENDER_DIR.exists(): shutil.rmtree(LOCAL_BLENDER_DIR)
subprocess.run(['tar','-xf',str(LOCAL_ARCHIVE),'-C',str(LOCAL)], check=True)
BLENDER = LOCAL_BLENDER_DIR / 'blender'
if not BLENDER.exists():
    raise RuntimeError('Blender 解压失败。')
print('✓ Blender 本地运行环境就绪')

print('\n④ 获取锁定版本的 v1 构建脚本…')
urllib.request.urlretrieve(SCRIPT_URL, LOCAL_SCRIPT)
print('✓ v1 script commit:', SCRIPT_COMMIT)

if LOCAL_OUT.exists(): shutil.rmtree(LOCAL_OUT)
LOCAL_OUT.mkdir(parents=True, exist_ok=True)

print('\n⑤ 云端 Blender 正在做 v1 解约束 + 髋宽预览…')
log_path = LOCAL_OUT / 'R_Master_v1_blender.log'
cmd = [
    'xvfb-run','-a',str(BLENDER),
    '--background',str(LOCAL_SOURCE),
    '--python',str(LOCAL_SCRIPT),
    '--','--out',str(LOCAL_OUT)
]
with log_path.open('w', encoding='utf-8') as log:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        log.write(line)
        if ('R Master v1' in line) or ('Error' in line) or ('Traceback' in line):
            print(line.rstrip())
    code = proc.wait()

if code != 0:
    tail = log_path.read_text(encoding='utf-8', errors='replace')[-8000:]
    print('\n--- Blender 日志末尾 ---\n' + tail)
    raise RuntimeError(f'Blender 运行失败，退出码 {code}。截图这一屏给二蛋。')

report_path = LOCAL_OUT / 'R_Master_v1_report.json'
preview_blend = LOCAL_OUT / 'R_Master_Align_v1_PREVIEW.blend'
imgs = [
    LOCAL_OUT / 'R_Master_v1_body_front.png',
    LOCAL_OUT / 'R_Master_v1_body_side.png',
    LOCAL_OUT / 'R_Master_v1_body_three_quarter.png'
]
required = [report_path, preview_blend, log_path, *imgs]
missing = [p.name for p in required if not p.exists()]
if missing:
    raise RuntimeError('缺少输出：' + ', '.join(missing))

report = json.loads(report_path.read_text(encoding='utf-8'))
acc = report.get('acceptance', {})
print('\n✓ R Master v1 BUILD_OK')
print('  骨骼数：', report.get('bone_count'))
print('  结构机械验收：', acc.get('mechanical_preview_pass'))
print('  最大骨段比例相对误差：', acc.get('primary_segment_ratio_max_relative_error'))
print('  髋宽：', report.get('hip_width'))
print('  Rest Pose 烘焙：', report.get('rest_pose_baked'))

print('\n⑥ 保存重文件到 Google Drive…')
# Keep latest deterministic and easy to find; no phone download needed for the 150+ MB blend.
for p in DRIVE_V1.iterdir():
    if p.is_file(): p.unlink()
for p in LOCAL_OUT.iterdir():
    if p.is_file(): shutil.copy2(p, DRIVE_V1 / p.name)
print('✓ 已保存到：MyDrive/R_Master/v1/latest/')

print('\n⑦ 灰模结构预览：')
for title, path in [('正面', imgs[0]), ('侧面', imgs[1]), ('3/4', imgs[2])]:
    display(Markdown(f'### {title}'))
    display(Image(filename=str(path), width=500))

print('\n⑧ 生成轻量 review 包（只含三张图 + report + log）…')
review_zip = LOCAL / 'R_Master_v1_Review.zip'
if review_zip.exists(): review_zip.unlink()
with zipfile.ZipFile(review_zip, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for p in [*imgs, report_path, log_path]:
        z.write(p, arcname=p.name)
shutil.copy2(review_zip, DRIVE_V1 / review_zip.name)
print(f'✓ Review 包：{review_zip.stat().st_size/1024/1024:.1f} MiB')
print('重的 .blend 已留在 Drive；只下载这个小 review 包给二蛋看。')
files.download(str(review_zip))
